# Tamil Nadu 9BA hourly peak-week unit commitment

This notebook loads the repository's nine-balancing-area FY2025-26 model, selects the Monday-Sunday week containing the statewide annual demand peak, solves unit commitment with HiGHS, checks nodal balances and transmission loading, and exports the solved network.

No equipment faults, forced outages, contingencies, or short-circuit events are introduced.

In [ ]:
from pathlib import Path

import pandas as pd
import pypsa

ROOT = Path.cwd()
if not (ROOT / "model").exists():
    ROOT = ROOT.parent

INPUT_DIR = ROOT / "model"
RESULT_FILE = ROOT / "results" / "tamil_nadu_9ba_2025_26.nc"

n_year = pypsa.Network(INPUT_DIR)

electrical_buses = n_year.buses.index[n_year.buses.carrier == "AC"]
transmission_links = n_year.links.index[n_year.links.carrier == "AC"]
assert len(electrical_buses) == 9, f"Expected 9 electrical buses, found {len(electrical_buses)}"
assert len(transmission_links) == 18, f"Expected 18 transmission corridors, found {len(transmission_links)}"
assert len(n_year.snapshots) == 8760
assert n_year.generators.committable.any(), "No committable generators found"
assert (n_year.snapshots.to_series().diff().dropna() == pd.Timedelta(hours=1)).all()

annual_demand = n_year.loads_t.p_set.sum(axis=1)
peak_timestamp = annual_demand.idxmax()
week_start = peak_timestamp.normalize() - pd.Timedelta(days=peak_timestamp.weekday())
week_end = week_start + pd.Timedelta(days=7)
week = n_year.snapshots[
    (n_year.snapshots >= week_start) & (n_year.snapshots < week_end)
]
assert len(week) == 168, f"Expected 168 hourly snapshots, found {len(week)}"

n = n_year.copy(snapshots=week)
import sys
sys.path.insert(0, str(ROOT / 'scripts'))
from nuclear_energy_targets import prepare_nuclear_targets, add_nuclear_targets, validate_nuclear_targets
prepare_nuclear_targets(n)

OIL_GAS_BUDGET_CONSTRAINT = "observed_fy2025_26_oil_gas_generation_target"
oil_gas_budget = (
    pd.read_csv(INPUT_DIR / "oil_gas_daily_energy_budget.csv", parse_dates=["date"])
    .set_index("date")
)
week_dates = pd.DatetimeIndex(week).normalize().unique()
week_oil_gas_target_mwh = float(
    oil_gas_budget.loc[week_dates, "daily_energy_target_mwh"].sum()
)
assert OIL_GAS_BUDGET_CONSTRAINT in n.global_constraints.index
n.global_constraints.at[
    OIL_GAS_BUDGET_CONSTRAINT, "constant"
] = week_oil_gas_target_mwh

def add_daily_oil_gas_targets(network, snapshots):
    add_nuclear_targets(network, snapshots)
    gas_units = network.generators.index[network.generators.carrier == "oil_gas"]
    dispatch = network.model["Generator-p"]
    normalized = pd.DatetimeIndex(snapshots).normalize()
    for day in normalized.unique():
        day_snapshots = snapshots[normalized == day]
        daily_target_mwh = float(oil_gas_budget.at[day, "daily_energy_target_mwh"])
        daily_dispatch_mwh = dispatch.sel(
            name=gas_units, snapshot=day_snapshots
        ).sum()
        network.model.add_constraints(
            daily_dispatch_mwh,
            "==",
            daily_target_mwh,
            name=f"OilGasDailyTarget-{day:%Y%m%d}",
        )

print(f"Loaded {len(electrical_buses)} balancing areas and {len(transmission_links)} corridors")
print(f"Committable generators: {int(n_year.generators.committable.sum())}")
print(f"Annual peak: {annual_demand.loc[peak_timestamp]:,.2f} MW at {peak_timestamp}")
print(f"Optimizing: {week_start} to {week_end - pd.Timedelta(hours=1)}")
print(f"Oil-and-gas target for selected week: {week_oil_gas_target_mwh / 1_000.0:,.2f} MU")

In [ ]:
n

## Model overview

Installed generator and storage capacities are shown by balancing area and carrier. Transmission limits are fixed and bidirectional. These tables describe the intact network; no fault-related availability modifications are made.

In [ ]:
generator_capacity = (
    n.generators.groupby(["bus", "carrier"]).p_nom.sum()
    .rename("capacity_MW")
    .to_frame()
)
storage_capacity = (
    n.storage_units.groupby(["bus", "carrier"]).p_nom.sum()
    .rename("capacity_MW")
    .to_frame()
)
corridors = n.links[
    ["bus0", "bus1", "p_nom", "p_min_pu", "efficiency"]
].rename(columns={"p_nom": "capacity_MW"})

display(generator_capacity)
display(storage_capacity)
display(corridors)

## Solve the peak week

This is a mixed-integer unit-commitment optimization. Full-year inputs remain available in the CSV model, but only the 168-hour peak week is solved here.

In [ ]:
status, termination_condition = n.optimize(
    solver_name="highs",
    solver_options={"mip_rel_gap": 1e-2, "log_to_console": False},
    extra_functionality=add_daily_oil_gas_targets,
    include_objective_constant=False,
)

if termination_condition != "optimal":
    raise RuntimeError(
        f"Optimization failed: {status}, {termination_condition}"
    )

print(f"Optimization: {status} ({termination_condition})")

In [ ]:
print(f'Nuclear maximum daily deviation: {validate_nuclear_targets(n):.3e} MWh')
weights = n.snapshot_weightings.generators
unserved_units = n.generators.index[
    n.generators.carrier == "unserved_energy"
]
unserved_mwh = (
    n.generators_t.p[unserved_units]
    .mul(weights, axis=0)
    .sum()
    .sum()
)
gas_units = n.generators.index[n.generators.carrier == "oil_gas"]
gas_daily_mwh = (
    n.generators_t.p[gas_units].sum(axis=1).mul(weights).groupby(n.snapshots.normalize()).sum()
)
gas_daily_targets_mwh = oil_gas_budget.loc[gas_daily_mwh.index, "daily_energy_target_mwh"]
max_gas_target_deviation_mwh = float((gas_daily_mwh - gas_daily_targets_mwh).abs().max())
assert max_gas_target_deviation_mwh < 1e-5

nodal_balance_residual_mw = n.buses_t.p.sum(axis=1).abs().max()
link_loading = n.links_t.p0.abs().div(n.links.p_nom, axis=1)
max_link_loading = link_loading.max().max()
most_loaded_link = link_loading.max().idxmax()
most_loaded_time = link_loading[most_loaded_link].idxmax()

print(f"Objective: {n.objective:,.2f}")
print(f"Unserved energy: {unserved_mwh:,.6f} MWh")
print(f"Oil-and-gas generation: {gas_daily_mwh.sum() / 1_000.0:,.2f} MU")
print(f"Maximum daily oil-and-gas target deviation: {max_gas_target_deviation_mwh:.3e} MWh")
print(f"Maximum nodal-balance residual: {nodal_balance_residual_mw:,.3e} MW")
print(
    f"Maximum corridor loading: {100 * max_link_loading:.2f}% "
    f"on {most_loaded_link} at {most_loaded_time}"
)

assert nodal_balance_residual_mw < 1e-5
assert max_link_loading <= 1.0 + 1e-6

n.export_to_netcdf(RESULT_FILE)
print(f"Exported solved network to {RESULT_FILE}")